# Exercise 3: A coverage baseline

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Generation and Dialogue · version 0.1.0*

> Compute analyser coverage on 1,000 sentences of human-written news text and on model-written news in the same language. How large is the difference, and what kinds of words account for it?

**Setup.** `pip install llama-cpp-python fasttext-wheel requests "turkicnlp[hfst]==0.5.0"` (the compiled `hfst` package needs Python 3.11 or 3.12). The generator is **Qwen2.5-1.5B-Instruct** in 8-bit GGUF form, run on the CPU with llama.cpp: set `QWEN_GGUF` to the file. The book's run converted the Hugging Face checkpoint at revision `989aa79` with llama.cpp's `convert_hf_to_gguf.py --outtype q8_0` (SHA-256 recorded in the outputs); Qwen's own `Qwen2.5-1.5B-Instruct-GGUF` q8_0 file is an equivalent starting point. Language identification uses GlotLID v3 (`GLOTLID_PATH`, otherwise downloaded from the Hub), morphology the Apertium analysers that come with TurkicNLP. Decoding is greedy (temperature 0) and every model reply is cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so the outputs can be inspected and a rerun is exact. **One small model is used throughout.** Its errors illustrate the measurement methods; they are not a ranking of models or a statement about larger ones.

Human text: the first 1,000 sentences of the KazNERD test split, television news (CC BY 4.0). Model text: short news items written by the model for 200 prompts (20 topics × 10 places), split into sentences, at most 1,000. Coverage is the share of alphabetic word tokens with at least one Apertium reading.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_test.txt"
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

# ---- the generator, with every reply cached ---------------------------------------------------
_LLM = {}
REPLIES = CACHE / "ch14-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["reply"]

def chat(messages, max_tokens=200):
    """Greedy reply of Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) to a list of chat messages."""
    key = hashlib.sha1(json.dumps([messages, max_tokens], ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        if "m" not in _LLM:
            from llama_cpp import Llama
            _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
        out = _LLM["m"].create_chat_completion(messages=messages, temperature=0.0, max_tokens=max_tokens)
        _CACHED[key] = out["choices"][0]["message"]["content"].strip()
        REPLIES.parent.mkdir(parents=True, exist_ok=True)
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "messages": messages, "reply": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

# ---- automatic fidelity signals ---------------------------------------------------------------
_LID = {}
def lid(text):
    """Unrestricted GlotLID v3 label and probability of one line of text."""
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

SENT = re.compile(r"(?<=[.!?…])\s+|\n+")
def sentences(text):
    return [s.strip() for s in SENT.split(text) if len(re.findall(r"[^\W\d_]", s)) >= 3]

def script_shares(text):
    letters = [c for c in text if c.isalpha()]
    count = collections.Counter("Cyrl" if "CYRILLIC" in unicodedata.name(c, "") else
                                "Latn" if "LATIN" in unicodedata.name(c, "") else "other" for c in letters)
    return {k: v / len(letters) for k, v in count.items()} if letters else {}

WORD = re.compile(r"[^\W\d_]+")
_FST = {}
def lookup(lang):
    """Raw Apertium analyser of TurkicNLP for kaz (Cyrillic) or uzb (Latin): word -> [(reading, weight)]."""
    if lang not in _FST:
        import hfst, turkicnlp
        from turkicnlp.resources.registry import ModelRegistry
        script = {"kaz": "Cyrl", "uzb": "Latn"}[lang]
        turkicnlp.download(lang, processors=["morph"])
        path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.automorf.hfst"
        fst = hfst.HfstInputStream(str(path)).read()
        _FST[lang] = (lambda w: fst.lookup(w) or fst.lookup(w.lower())), sha256(path)
    return _FST[lang][0]

def accepted(word, lang):
    return any(r and not r.startswith("*") and math.isfinite(w) for r, w in lookup(lang)(word))

def coverage(texts, lang):
    words = [w for t in texts for w in WORD.findall(t)]
    ok = sum(accepted(w, lang) for w in words)
    return ok, len(words)

print("generator:", Path(QWEN_GGUF).name, "| sha256", sha256(QWEN_GGUF)[:16] if Path(QWEN_GGUF).exists() else "missing")

In [ ]:
# Human-written news: the first 1,000 sentences of the KazNERD test split (television news text, CC BY 4.0).
human, cur = [], []
for line in fetch(KAZNERD).read_text(encoding="utf-8").splitlines():
    if line.strip():
        cur.append(line.split()[0])
    elif cur:
        human.append(" ".join(cur)); cur = []
        if len(human) == 1000:
            break
TOPICS = ["ауыл шаруашылығы", "білім беру", "денсаулық сақтау", "спорт", "мәдениет", "экономика", "ауа райы", "көлік",
          "туризм", "экология", "ғылым", "құрылыс", "энергетика", "сауда", "жастар саясаты", "су ресурстары", "өнеркәсіп",
          "цифрландыру", "әлеуметтік көмек", "мерекелер"]
REGIONS = ["Алматы", "Астана", "Шымкент", "Қарағанды", "Ақтөбе", "Атырау", "Павлодар", "Түркістан облысы", "Семей", "Қостанай"]
generated, prompts = [], 0
for topic in TOPICS:
    for region in REGIONS:
        if len(generated) >= 1000:
            break
        reply = chat([{"role": "system", "content": "You write short television news items in Kazakh (Cyrillic script)."},
                      {"role": "user", "content": f"{region} қаласындағы {topic} саласындағы жаңалық туралы бес сөйлемнен тұратын қысқа хабар жазыңыз."}],
                     max_tokens=220)
        prompts += 1
        generated += sentences(reply)
generated = generated[:1000]
print(f"human: {len(human)} KazNERD sentences; model: {len(generated)} sentences from {prompts} prompts")
for name, sents in (("human", human), ("model", generated)):
    ok, n = coverage(sents, "kaz")
    labs = collections.Counter(lid(s)[0] for s in sents)
    kk = [s for s in sents if lid(s)[0] == "kaz_Cyrl"]
    ok2, n2 = coverage(kk, "kaz")
    print(f"{name}: analyser coverage {ok:,}/{n:,} = {ok / n:.1%}; GlotLID kaz_Cyrl {labs['kaz_Cyrl'] / len(sents):.1%}, "
          f"other labels {dict(labs.most_common(4)[1:])}; coverage within kaz_Cyrl sentences {ok2 / n2:.1%}")
def unknown(sents):
    return collections.Counter(w for s in sents for w in WORD.findall(s) if not accepted(w, "kaz"))
for name, sents in (("human", human), ("model", generated)):
    u = unknown(sents)
    print(f"most frequent unanalysed words, {name}: " + ", ".join(f"{w} ({c})" for w, c in u.most_common(25)))

## Worked solution

Output of `experiments/ch14_exercises/run_ch14_exercises.py`:

```
human: 1000 KazNERD sentences; model: 668 sentences from 200 prompts
human: analyser coverage 8,798/9,309 = 94.5%; GlotLID kaz_Cyrl 99.0%, other labels {'rus_Cyrl': 3, 'kir_Cyrl': 3, 'kaa_Cyrl': 2}; coverage within kaz_Cyrl sentences 94.6%
model: analyser coverage 5,562/6,611 = 84.1%; GlotLID kaz_Cyrl 93.7%, other labels {'kir_Cyrl': 18, 'kaa_Cyrl': 9, 'anp_Deva': 9}; coverage within kaz_Cyrl sentences 84.7%
most frequent unanalysed words, human: і (38), іі (27), нші (16), ыы (13), ншы (7), көпбалалы (7), ке (6), еуі (5), ы (4), панельдік (4), аффинаж (3), депозиттік (3), Даже (3), нетпесе (3), пәленбай (3), халге (3), уже (3), қолжетімді (3), нарқына (3), атсалысты (3), Қарабалық (3), інен (3), акиматтан (2), қоластындағы (2), с (2)
most frequent unanalysed words, model: Берилген (171), құрылышы (66), өзгөрүшү (50), өзгөрүшүн (47), Бул (40), сөйлемдің (37), өзгөрүлөрдү (34), өзгөрүші (23), Аныктаңыз (18), өзгөрдік (15), өнүк (15), қ (14), хабардын (14), өзгөрүлдіктерінің (14), өзгөрүлөрдің (13), өзгөрүлүштерінің (13), өзгөрді (12), цифрландыру (12), қорытындықтардың (11), хабартардың (11), Аны (10), өзгөрүлүштерін (10), құрылуу (10), құрылышынан (9), жөнүндө (8)
```

**Coverage differs by 10.4 percentage points.** Apertium accepts 94.5% of 9,309 tokens in 1,000 KazNERD sentences and 84.1% of 6,611 tokens in 668 model sentences. Restricting to GlotLID's Kazakh-labelled sentences gives 94.6% and 84.7%. A sentence label does not exclude mixed-language words.

**Inspect frequent rejections.** Model tokens include *Берилген* (171 occurrences), *өзгөрүшү* and *Бул*; human-text tokens include *і*, *іі*, *нші* and *көпбалалы*. Repetition increases a token's weight, so also report distinct forms and inspect their contexts.

The model sample comes from 200 news-style prompts and is not matched to the human sample for content or length. Neither a rejection nor its source establishes whether a word is erroneous. Independent annotation is needed to separate language mixing, malformed words, tokenisation artefacts and gaps in the analyser.

## Your turn — extension

1. Compute coverage separately for each of the 20 topics and see whether some drift more.
2. Add Kyrgyz forms to a list and count how many rejected model words they explain.